In [ ]:
#Use GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
#Prompt that specifies to extract information regarding each of the academic headlines and to populate only those.
SYSTEMP_PROMPT = """
You are an expert academic text analyzer with the ability to understand complex language and structure. Your task is to extract and organize content from a provided text file into a new text, sectioned into only the five categories.
Follow these instructions carefully, ensuring each section aligns strictly with its academic definition and provides clarity and relevance:

Populate these sections given the instructions:
1. Introduction:
   - Identify and summarize content that sets the stage for the text, including the context, background, objectives, or rationale.
   - Use concise academic language to convey the essence of the introduction.

2. Methodology:
   - Extract details about the processes, approaches, or techniques described in the text.
   - Ensure the focus remains on "how" the work was conducted, avoiding results or conclusions.

3. Theory:
   - Locate and articulate the theoretical frameworks, key concepts, or models referenced.
   - Highlight abstract ideas and principles central to the document's intellectual framework.

4. Results:
   - Summarize the key findings or data presented.
   - Prioritize clarity and ensure the extracted information represents the outcomes accurately.

5. **Conclusion**:
   - Extract and synthesize the final thoughts, interpretations, and implications provided.
   - Aim for a cohesive summary of the document's end message.

Pleae be thorough and include as much text as you can while still being relevant. This is the key objective of the task. To include as much as possible while still relevant to the section. Do not ever write "We" or "I", instead refer to the authors as the authors.



BE AWARE OF:
   - Recognize subtle variations in language that may indicate sections (e.g., “we propose” often signals objectives, “the study employed” indicates methodology, etc.).
   - Use transitions or signposts (e.g., “in conclusion,” “the results showed”) as cues for section identification.

   - If a section appears missing or incomplete, explicitly state: “No content available for this section.”
   - Avoid redundancy or inclusion of unrelated content.

   - Use clear section headers (e.g., **Introduction**, **Methodology**, etc.).
   - Ensure each section is formatted in plain text with no additional formatting or markdown.

   - Do not speculate or infer information beyond what is present in the text.
   - Extract only verbatim or appropriately paraphrased information relevant to each section.

   - Before extracting each section, silently validate the input text to identify explicit or implied section markers.
   - When information spans multiple sections, prioritize the most relevant alignment.

Your output should ALWAYS be structured as follows:
Introduction:
[Extracted content or "No content available for this section."]

Methodology:
[Extracted content or "No content available for this section."]

Theory:
[Extracted content or "No content available for this section."]

Results:
[Extracted content or "No content available for this section."]

Conclusion:
[Extracted content or "No content available for this section."]

Focus on delivering a coherent and logically structured response tailored to the given text file. Adhere strictly to these instructions for optimal performance.

"""

In [ ]:
#We use 3.1-8B, which is the cheaper of the 3.1 models.
MODEL = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
#Imports
import torch
from accelerate import Accelerator
import transformers
import pickle

from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

In [ ]:
#A fail safe way to read the file into string that considers encoding and error handling
def read_file_to_string(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read()
        return content
    except UnicodeDecodeError:
        encodings = ['latin-1', 'cp1252', 'iso-8859-1']
        for encoding in encodings:
            try:
                with open(filename, 'r', encoding=encoding) as file:
                    content = file.read()
                print(f"Successfully read file using {encoding} encoding.")
                return content
            except UnicodeDecodeError:
                continue

        print(f"Error: Could not decode file '{filename}' with any common encoding.")
        return None
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return None
    except IOError:
        print(f"Error: Could not read file '{filename}'.")
        return None

In [ ]:
#We use the extracted file of the previous step as input
INPUT_PROMPT = read_file_to_string('clean_extracted_text.txt')

In [ ]:
from transformers import pipeline
import torch
from tqdm import tqdm
import time

#Sets up the generation pipeline with gpu and chosen model.
print("Initializing text-generation pipeline...")
pipeline = transformers.pipeline(
    "text-generation",
    model=MODEL,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map='auto',
)
print("Pipeline initialized successfully.")

#Prompts instructions with system prompt and input file
messages = [
    {"role": "system", "content": SYSTEMP_PROMPT},
    {"role": "user", "content": INPUT_PROMPT},
]

#Shows progress
print("Starting text generation...")
with tqdm(total=1, desc="Text generation progress", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}") as pbar:
    try:
        outputs = pipeline(
            messages,
            max_new_tokens=8126,
            temperature=1,
        )
        pbar.update(1)
        print("Text generation complete.")
    except Exception as e:
        pbar.close()
        print(f"An error occurred during text generation: {e}")


Initializing text-generation pipeline...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Pipeline initialized successfully.
Starting text generation...


Text generation progress:   0%|          | 0/1Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Text generation progress: 100%|██████████| 1/1

Text generation complete.


In [ ]:
#Inspect the generated text
save_string_pkl = outputs[0]["generated_text"][-1]['content']
print(outputs[0]["generated_text"][-1]['content'])

**Introduction:**

Disruptive innovation has gained significant attention among practitioners and researchers, but its core principles remain misunderstood. Despite the prevalence of disruption theory, subsequent empirical research has rarely engaged with its key theoretical arguments. This article aims to reinvigorate academic interest in disruptive innovation by proposing underexplored topics and synthesizing existing research. The authors propose a comprehensive conceptual framework for understanding disruptive innovation, building on academic journals, practitioner publications, and books. They also identify three novel areas of research: response strategies, performance trajectories, and innovation metrics.

**Methodology:**

The authors followed a systematic approach to review relevant literature on the origins of disruptive innovation. They searched the Web of Science database for articles citing Bower and Christensen (1995), Christensen and Bower (1996), or Christensen (1997) t

In [ ]:
#Save as output.txt to use for next step.
file_path = "output.txt"
with open(file_path, "w", encoding="utf-8") as file:
    file.write(save_string_pkl)

print(f"String saved to {file_path}")

String saved to output.txt
